In [1]:
import argparse
import json
import math
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

In [2]:
@dataclass(frozen=True)
class ForecastWindow:
    """One forecasting window, uniquely identified by origin + horizon."""

    window_id: str
    origin_id: str
    origin_idx: int
    history_start_idx: int
    history_end_idx: int       # exclusive
    target_start_idx: int
    target_end_idx: int        # exclusive

@dataclass(frozen=True)
class Demonstration:
    """A past example used in a few-shot prompt."""

    demo_id: str
    input_start_idx: int
    input_end_idx: int
    target_start_idx: int
    target_end_idx: int

In [ ]:
def candidate_demonstrations(
    df: pd.DataFrame,
    window: ForecastWindow,
    lookback: int,
    horizon: int,
    min_gap: int = 0,
) -> List[Demonstration]:
    """Return demonstrations whose complete target was known at forecast time.

    For a current forecast origin O, a demonstration is valid when its
    complete target ends at or before O. Its target may therefore overlap
    the current *historical input* window. That is not leakage: those target
    values were already observed by time O.

    `min_gap` optionally enforces extra separation between the demonstration
    target end and the current forecast origin.
    """
    candidates: List[Demonstration] = []

    # target_end is exclusive. A target ending exactly at O is fully known
    # when the forecast at O is made.
    latest_allowed_target_end = window.origin_idx - min_gap

    for demo_origin in range(lookback, latest_allowed_target_end + 1):
        demo_input_start = demo_origin - lookback
        demo_input_end = demo_origin
        demo_target_start = demo_origin
        demo_target_end = demo_origin + horizon

        if demo_target_end > latest_allowed_target_end:
            continue

        demo_origin_id = df.iloc[demo_origin][DATE].strftime("%Y%m%d")
        demo_id = f"{demo_origin_id}_h{horizon}"

        candidates.append(
            Demonstration(
                demo_id=demo_id,
                input_start_idx=demo_input_start,
                input_end_idx=demo_input_end,
                target_start_idx=demo_target_start,
                target_end_idx=demo_target_end,
            )
        )

    return candidates

df = pd.read_csv("/home/wagner-barbosa/Documents/promptcast_forecast/data/processed/joinville_processed_1day.csv")

